# `trained_model_xai.py` — Explainability (XAI) Bridge

## Purpose
Provides the explainability methods for the trained model that the website's XAI panel calls. Returns results in the same JSON shape as the legacy `ClearViewExplainer` so the frontend doesn't need to change.

## XAI Methods

| Method | Speed | What it shows |
|--------|-------|---------------|
| `explain_ig_aspect()` | Fast | Which tokens most influenced the prediction for a specific aspect (via Integrated Gradients) |
| `explain_ig_conflict()` | Medium | Which tokens simultaneously drive BOTH positive and negative aspects (conflict drivers) |
| `explain_msr_delta()` | Fast | Probability distribution before vs. after temperature scaling |
| `explain_lime_aspect()` | Slow | Word-level attribution via LIME (perturbs text, fits linear surrogate) |
| `explain_shap_aspect()` | Slow | Word-level attribution via SHAP (Shapley values from PartitionExplainer) |

## Frontend API Contract
All methods return dicts with `top_tokens: [[token, score], ...]` — positive scores mean the token supported the predicted label, negative mean it opposed it.

In [ ]:
import sys, os, re

# ── Path setup ─────────────────────────────────────────────────────────────────
current_dir   = os.path.dirname(os.path.abspath(__file__))
project_root  = os.path.abspath(os.path.join(current_dir, '..', '..', '..'))
inference_dir = os.path.join(project_root, 'ml-research', 'outputs', 'cosmetic_sentiment_v1', 'evaluation')
ml_src_dir    = os.path.join(project_root, 'ml-research', 'src')

for d in [inference_dir, ml_src_dir]:
    if d not in sys.path: sys.path.insert(0, d)

from inference import SentimentPredictor

In [ ]:
class TrainedModelXAI:
    """
    XAI wrapper providing explain_* methods compatible with the backend's expected API shape.
    """
    def __init__(self, checkpoint_path: str, temperature: float = 0.5):
        self.predictor    = SentimentPredictor(checkpoint_path, temperature=temperature)
        self.aspect_names = self.predictor.aspect_names

    # ─── IG-based Aspect Explanation ──────────────────────────────────────────
    def explain_ig_aspect(self, text: str, aspect: str, enable_msr=True, top_k=10) -> dict:
        """
        Returns top-k tokens that most influenced the prediction for `aspect`
        using Integrated Gradients (Captum).

        n_steps=50: number of Riemann-sum steps along the baseline→input path.
        Higher steps = more accurate attribution but slower.
        """
        ig_res = self.predictor.explain_with_integrated_gradients(
            text=text, aspect=aspect, n_steps=50, top_k=top_k, save_path=None
        )

        # Clean tokens and sort by absolute attribution magnitude
        top_tokens = []
        for token, attr in zip(ig_res['tokens'], ig_res['attributions']):
            if token not in ('<s>', '</s>', '<pad>', '<mask>') and len(token.strip('Ġ▁ ')) > 0:
                top_tokens.append([self._clean_token(token), round(float(attr), 6)])

        # Sort by |score| descending, deduplicate preserving highest
        top_tokens.sort(key=lambda x: abs(x[1]), reverse=True)
        seen, deduped = set(), []
        for t, v in top_tokens:
            if t.lower() not in seen:
                seen.add(t.lower()); deduped.append([t, v])
            if len(deduped) >= top_k: break

        pred = self.predictor.predict(text, aspect)  # Get probability dict

        return {
            'top_tokens': deduped, 'method': 'attention',  # 'attention' literal keeps frontend routing working
            'task': f'aspect:{aspect}',
            'predicted'  : ig_res['target_label'],
            'confidence' : round(ig_res['confidence'], 4),
            'probs': {
                'negative': round(pred['probabilities']['negative'], 4),
                'neutral' : round(pred['probabilities']['neutral'],  4),
                'positive': round(pred['probabilities']['positive'], 4),
            }
        }

    # ─── Conflict Token Detection ──────────────────────────────────────────────
    def explain_ig_conflict(self, text: str, enable_msr=True, top_k=10) -> dict:
        """
        Identify tokens that simultaneously drive POSITIVE and NEGATIVE aspect predictions.
        Score for each token = sqrt(pos_attention_sum * neg_attention_sum).
        High score = that token is active in BOTH polarities = a conflict driver.
        """
        aspect_attentions = {}  # {aspect: {token: attention_weight}}
        predictions       = {}  # {aspect: 'positive' | 'neutral' | 'negative'}

        for asp in self.aspect_names:
            result = self.predictor.predict(text, asp, return_attention=True)
            predictions[asp] = result['sentiment']
            if 'attention' in result:
                # Build token→weight dict for this aspect
                aspect_attentions[asp] = dict(
                    zip(result['attention']['tokens'], result['attention']['weights'])
                )

        if not aspect_attentions:
            return {'top_tokens': [], 'method': 'attention', 'task': 'conflict'}

        all_tokens = {
            t for d in aspect_attentions.values() for t in d
            if not t.startswith('<') and t not in ('Ġ', '')
        }

        token_conflict = {}
        for tok in all_tokens:
            pos_sum = sum(aspect_attentions[asp].get(tok, 0.0)
                         for asp in aspect_attentions if predictions.get(asp) == 'positive')
            neg_sum = sum(aspect_attentions[asp].get(tok, 0.0)
                         for asp in aspect_attentions if predictions.get(asp) == 'negative')
            score = (pos_sum * neg_sum) ** 0.5  # Geometric mean: high only when BOTH sides are high
            if score > 0: token_conflict[tok] = round(score, 6)

        sorted_tokens = sorted(token_conflict.items(), key=lambda x: x[1], reverse=True)
        top_tokens    = [[self._clean_token(t), s] for t, s in sorted_tokens[:top_k]]

        return {'top_tokens': top_tokens, 'method': 'attention', 'task': 'conflict'}

    # ─── MSR Delta (Before/After Temperature Scaling) ─────────────────────────
    def explain_msr_delta(self, text: str, aspect: str, top_k=10) -> dict:
        """
        Shows probability distribution at temperature=1.0 ('before') vs. the
        calibrated temperature ('after'). Illustrates the effect of temperature scaling.
        """
        raw_result  = self.predictor.predict(text, aspect)
        probs_after = [raw_result['probabilities'][k] for k in ('negative', 'neutral', 'positive')]

        original_temp = self.predictor.temperature
        self.predictor.temperature = 1.0  # Disable scaling temporarily
        try:
            before_result = self.predictor.predict(text, aspect)
            probs_before  = [before_result['probabilities'][k] for k in ('negative', 'neutral', 'positive')]
        finally:
            self.predictor.temperature = original_temp  # Always restore

        return {
            'prob_before': [round(p, 4) for p in probs_before],
            'prob_after' : [round(p, 4) for p in probs_after],
            'method'     : 'msr_delta',
            'task'       : f'aspect:{aspect}',
        }

    # ─── LIME Explanation ─────────────────────────────────────────────────────
    def explain_lime_aspect(self, text: str, aspect: str, num_samples=100, top_k=10) -> dict:
        """
        LIME: generates perturbed text samples, predicts on all of them,
        then fits a local linear model to find which words drove the prediction.
        Slower than attention-based methods but fully model-agnostic.
        """
        from lime.lime_text import LimeTextExplainer
        import numpy as np

        result         = self.predictor.predict(text, aspect)
        pred_sentiment = result['sentiment']
        labels_map     = {'negative': 0, 'neutral': 1, 'positive': 2}
        target_idx     = labels_map.get(pred_sentiment, 2)

        def predictor_fn(texts):
            """LIME calls this with a list of perturbed text strings."""
            return np.array([[res['probabilities']['negative'], res['probabilities']['neutral'], res['probabilities']['positive']]
                             for res in [self.predictor.predict(t, aspect) for t in texts]])

        explainer = LimeTextExplainer(class_names=['negative', 'neutral', 'positive'])
        exp        = explainer.explain_instance(text, predictor_fn, labels=(target_idx,),
                                               num_features=top_k, num_samples=num_samples)
        top_tokens = [[word, round(float(score), 6)] for word, score in exp.as_list(label=target_idx)]

        return {
            'top_tokens': top_tokens, 'method': 'lime', 'task': f'aspect:{aspect}',
            'predicted': pred_sentiment, 'confidence': round(result['confidence'], 4),
        }

    # ─── SHAP Explanation ─────────────────────────────────────────────────────
    def explain_shap_aspect(self, text: str, aspect: str, max_evals=100, top_k=10) -> dict:
        """
        SHAP PartitionExplainer: assigns each word a Shapley value indicating
        how much it contributed to the predicted class. Uses whitespace tokenisation.
        """
        import shap, numpy as np
        result = self.predictor.predict(text, aspect)
        labels_map = {'negative': 0, 'neutral': 1, 'positive': 2}
        target_idx = labels_map.get(result['sentiment'], 2)

        def predictor_fn(texts):
            return np.array([[res['probabilities']['negative'], res['probabilities']['neutral'], res['probabilities']['positive']]
                             for res in [self.predictor.predict(str(t), aspect) for t in texts]])

        masker   = shap.maskers.Text(r'\W')  # Whitespace-based word masking
        explainer = shap.Explainer(predictor_fn, masker, output_names=['negative', 'neutral', 'positive'])

        try:
            shap_values = explainer([text], max_evals=max_evals)
            tokens      = shap_values.data[0]
            values      = shap_values.values[0, :, target_idx]  # Shapley values for predicted class
            pairs = sorted([(t.strip(), round(float(v), 6)) for t, v in zip(tokens, values) if t.strip()],
                           key=lambda x: abs(x[1]), reverse=True)
            seen, top_tokens = set(), []
            for t, v in pairs:
                if t.lower() not in seen: seen.add(t.lower()); top_tokens.append([t, v])
                if len(top_tokens) >= top_k: break
        except Exception as e:
            print(f'[SHAP] Error: {e}'); top_tokens = []

        return {
            'top_tokens': top_tokens, 'method': 'shap', 'task': f'aspect:{aspect}',
            'predicted': result['sentiment'], 'confidence': round(result['confidence'], 4),
        }

    @staticmethod
    def _clean_token(token: str) -> str:
        """Strip RoBERTa BPE prefix character (Ġ = space in RoBERTa tokeniser)."""
        return token.lstrip('Ġ▁')